In [ ]:

# ==================================
# 1. Import Libraries
# ==================================
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor

# ==================================
# 2. Collect Data
# ==================================
df = sns.load_dataset("diamonds")

print(df.head())
print("\nPrice Statistics:\n", df["price"].describe())

# ==================================
# 3. Feature Engineering
# ==================================
# Create diamond volume
df["volume"] = df["x"] * df["y"] * df["z"]

# Remove unrealistic values
df = df[(df["x"] > 0) & (df["y"] > 0) & (df["z"] > 0)]

# Features and target
X = df.drop("price", axis=1)
y = df["price"]

# Column types
num_cols = ["carat","depth","table","x","y","z","volume"]
cat_cols = ["cut","color","clarity"]

# ==================================
# 4. Preprocessing
# ==================================
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# ==================================
# 5. Build Pipeline
# ==================================
model = GradientBoostingRegressor()

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

# ==================================
# 6. Hyperparameter Tuning
# ==================================
param_grid = {
    "model__n_estimators": [200, 300],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 5]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
grid.fit(X_train, y_train)

# ==================================
# 7. Evaluate Model
# ==================================
best_model = grid.best_estimator_

predictions = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("\nBest Parameters:", grid.best_params_)
print("RMSE:", rmse)
print("R2 Score:", r2)

# ==================================
# 8. Iteration (Further improvement)
# ==================================
model2 = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5
)

pipeline2 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model2)
])

pipeline2.fit(X_train, y_train)

pred2 = pipeline2.predict(X_test)

print("\nImproved R2 Score:", r2_score(y_test, pred2))

   carat      cut color clarity  depth  table  price     x     y     z
0   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
1   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
2   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
3   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23  2.63
4   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35  2.75

Price Statistics:
 count    53940.000000
mean      3932.799722
std       3989.439738
min        326.000000
25%        950.000000
50%       2401.000000
75%       5324.250000
max      18823.000000
Name: price, dtype: float64


KeyboardInterrupt: 